In [ ]:
# TITLE
# RadarNetCDFloader.ipynb
# TITLE

import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

# import zipfile as zp          # used for unzipping ppi files
from pathlib import Path      # used to play with pathnames to save 
# from datetime import datetime # used to manipulate time :)

# import wradlib as wr          # used for having fun with radar data

from PIL import Image         # used for creating gif loops
import os                     # used for retrieving file names

# import h5py                   # used for reading .h5 files (Radar Level 1 data)
# import h5netcdf               # used for converting .h5 files to NetCDF

# # TAKEN FROM "Part4IntroductionToGridding"
# import cartopy.crs as ccrs
# import pyart
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# import matplotlib.ticker as mticker

# # SPECIAL METHOD TO IMPORT LEROI RADAR GRIDDING PACKAGE FROM LOCAL DIRECTORY
import sys
# sys.path.append('/home/563/sg3241/Notebooks')
# from leroi.leroi import *

In [ ]:
# FUNCTION
#pcolormesh but takes 1D X and Y coordinates for centres of the pixels

def pcolormeshC(x_centers, y_centers, z, ax=None,
                            shading='auto', **pcolor_kwargs):
    """
    Create a pcolormesh from a 2D array and 1D coordinate-center arrays.

    Parameters
    ----------
    x_centers : 1D array
        X coordinates of cell centers (length = number of columns in z)
    y_centers : 1D array
        Y coordinates of cell centers (length = number of rows in z)
    z : 2D array
        Data array with shape (len(y_centers), len(x_centers))
    ax : matplotlib.axes.Axes, optional
        Existing axis to draw on
    shading : str
        Passed to pcolormesh (default: 'auto')
    **pcolor_kwargs
        Extra kwargs passed to pcolormesh

    Returns
    -------
    pcm : QuadMesh
        The pcolormesh object
    """

    x_centers = np.asarray(x_centers)
    y_centers = np.asarray(y_centers)
    z = np.asarray(z)

    if z.shape != (len(y_centers), len(x_centers)):
        raise ValueError(
            f"z shape {z.shape} does not match "
            f"(len(y_centers), len(x_centers)) = "
            f"({len(y_centers)}, {len(x_centers)})"
        )

    # Convert centers -> edges
    def centers_to_edges(c):
        dc = np.diff(c)

        edges = np.empty(len(c) + 1)

        # Interior edges
        edges[1:-1] = c[:-1] + dc / 2

        # Extrapolate outer edges
        edges[0] = c[0] - dc[0] / 2
        edges[-1] = c[-1] + dc[-1] / 2

        return edges

    x_edges = centers_to_edges(x_centers)
    y_edges = centers_to_edges(y_centers)

    if ax is None:
        fig, ax = plt.subplots()

    pcm = ax.pcolormesh(
        x_edges,
        y_edges,
        z,
        shading=shading,
        **pcolor_kwargs
    )

    ax.set_xlabel("X")
    ax.set_ylabel("Y")

    return pcm

In [ ]:
# VERTICAL CROSS SECTION PLOTTING
# THIS BLOCK LOADS IN FROM NET CDF FILES STORED IN SCRATCH

# THIS BLOCK IS WHERE THE USER PUTS INFO ABOUT THE RADAR

# PLEASE MAKE IT SO THE PLOT VARIABLES CAN BE CHOSEN HERE!!!! (ADD STRING EXECUTERS AND SUCH)
# PlotType = 'Vert'
# PlotVar  = 'corrected_reflectivity'
# Slice    = str(EWsliceKM) + 'kmEW'

# the reference number for the radar location (ie 22 is Mackay)
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)
GridOrPPI = 'ppi'

if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marabong'
else:
    RadarSiteName = 'Site ' + RadarIDno


# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 3
RadarDay   = 9

# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)

RadarFileDate  = YYYY + MM + DD 

houri = 12
mini = 50


# THIS BLOCK IS ABOUT THE VERTICAL CROSS SECTION SLICE NORTH OR SOUTH OF THE RADAR
# THIS CODE ASSUMES THE X AND Y GRIDS ARE EVERY 1 KM
# IT WILL HAVE TO BE REWRITTEN TO CONSIDER IN-BETWEEN VALUES

EWsliceKM = 20 # [km]     # number of km north or south of the radar you want to take the east-west slice for x-section
EWsliceNorS = 'North'    # direction ['North' or 'South'] from the radar you want the slice taken

# add a positive or negative sign to the slice for math
if (EWsliceNorS == 'South'):
    EWsliceKMsign = EWsliceKM * -1
elif (EWsliceNorS == 'North'):
    EWsliceKMsign = EWsliceKM * 1
else:
    print("Please choose EWsliceNorS to be 'North' or 'South'.")


# for houri in range(0,24):
#     for mini in range(0,60,5):
        
RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 

# add a string of format hh:mm:ss for printing
print('working on ' + RadarFileTimePrint)




xgrid = xr.open_dataset('/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/' + \
                        RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')

# THIS CODE ASSUMES THE X AND Y GRIDS ARE EVERY 1 KM
# IT WILL HAVE TO BE REWRITTEN TO CONSIDER IN-BETWEEN VALUES

yvalsKM = np.array(xgrid.x) * 0.001             # x coordinates in km

# farthest south and north y-values
MinSliceKM = int(np.min(yvalsKM))
MaxSliceKM = int(np.max(yvalsKM))

# for EWsliceKM in range(MinSliceKM, MaxSliceKM+1):
#     print('working on slice at ' + str(EWsliceKMsign) + ' km' )

# find the vertical cross section index in the coordinates
EWslicei = np.where(yvalsKM == EWsliceKMsign)[0][0]  # index in the x-coordinates where that north-south km value lives

# Create a vertical plot of the reflectivity for a 5-min period
    
fig, ax = plt.subplots(figsize=(8,6))
GridViewer = pcolormeshC(xgrid.x*0.001, xgrid.z*0.001, xgrid.corrected_specific_differential_phase[0,:,EWslicei,:], ax=ax, cmap='nipy_spectral', vmin=0, vmax=90)
# mutiply by 0.001 to get distances in km                                        # THIS SLICE COMES FROM 50 KM NORTH OF THE RADAR TO BETTER SEE THE VOLUME

ax.set_xlabel('East-West Distance [km]')
ax.set_ylabel('Altitude Above Radar [km]')
plt.title('Reflectivity Cross Section for ' + RadarSiteName + ' Radar\n For Slice Taken ' + \
      str(EWsliceKM) + ' km ' + EWsliceNorS + ' of the Radar on\n' + \
      RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
      str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC')
plt.colorbar(GridViewer, ax=ax, label = 'Reflectivity [dBZ]')
plt.grid()

plt.xlim([MinSliceKM, MaxSliceKM])
plt.ylim([0,20])

# SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/' + PlotType + '/' + RadarIDno + '/' + RadarFileDate + '/'
# SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + PlotVar + '_' + PlotType + Slice + '.png'

# SavePath = SaveFolder + SaveFile

# if not Path(SaveFolder).exists():
#     print('Creating Folder: ' + SaveFolder)
#     Path(SaveFolder).mkdir(parents=True, exist_ok=True)

# plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
# plt.close()

In [ ]:
# HORIZONTAL CROSS SECTION PLOTTING (LOADS IN FROM NET CDF FILES STORED IN SCRATCH)

# CHOOSE YOUR RADAR
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)

# name the radar site
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marabong'
else:
    RadarSiteName = 'Site ' + RadarIDno


# CHOOSE YOUR DAY
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 2
RadarDay   = 14

# CHOOSE YOUR ALTITUDE
Altitude = 2500  # [m] choose a multiple of 500 m to look at a CAPI for

# CHOOSE YOUR VARIABLE
# LIST OF POSSIBLE VARIABLES
# [Z]         'corrected_reflectivity'
# [CC]        'corrected_cross_correlation_ratio'
# [ZDR]       'corrected_differential_reflectivity'
# [KDP]       'corrected_specific_differential_phase'
# [PhiDP]     'corrected_differential_phase'
# [IntAtt]    'path_integrated_attenuation'
# [DifIntAtt] 'path_integrated_differential_attenuation'
# [EchClas]   'radar_echo_classification'
# [V]         'corrected_velocity'
# [AzSh]      'azshear'

Var = 'ZDR'

if (Var == 'Z'):
    VarName     = 'Reflectivity'
    VarNameLong = 'corrected_reflectivity'
    VarMinVal = -10 # [dBZ]
    VarMaxVal =  60 # [dBZ]
    VarColourBar = 'nipy_spectral'
elif (Var == 'ZDR'):
    VarName     = 'Differential Reflectivity'
    VarNameLong = 'corrected_differential_reflectivity'
    VarMinVal = -5 # [dB]
    VarMaxVal =  5 # [dB]
    VarColourBar = 'RdBu'

# CHOOSE YOUR HOUR and MINUTE
houri = 12
mini = 00

# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)

# write out the data in one string
RadarFileDate  = YYYY + MM + DD 

RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 

# add a string of format hh:mm:ss for printing
print('working on ' + RadarFileTimePrint)
xgrid = xr.open_dataset('/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/' + \
                        RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')

alti = np.where(xgrid.z == Altitude) # this is a double nested array for some reason

if ( np.size(alti) != 1): 
    print( str(Altitude) + ' m is not a valid altitude in the data')
else:
    alti = alti[0][0] # take the index out of the double nested array

    # for houri in range(0,24):
    #     for mini in range(0,60,5):
    
    fig, ax = plt.subplots(figsize=(8,6))
    GridViewer = pcolormeshC(xgrid.x*0.001, xgrid.y*0.001, xgrid[VarNameLong][0,alti,:,:], ax=ax, cmap=VarColourBar , vmin=VarMinVal, vmax=VarMaxVal)
    # mutiply by 0.001 to get distances in km                                        # [the only one time, elevation, y-coords, x-coords]
    
    ax.set_xlabel('East-West Distance [km]')
    ax.set_ylabel('North-South Distance [km]')
    plt.title(VarName + ' for ' + RadarSiteName + ' Radar\nat ' + str(Altitude) + ' m Altitude\non ' + \
              RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
              str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC')
    plt.colorbar(GridViewer, ax=ax, label = 'Reflectivity [dBZ]')
    plt.grid()

    # ADD LABELS FOR ALTITUDE AND SUCH
    # ADD LABELS FOR ALTITUDE AND SUCH
    # ADD LABELS FOR ALTITUDE AND SUCH
    
    # SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '/'
    # SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + GridOrPPI + '500m.png'
    
    # SavePath = SaveFolder + SaveFile
    
    # if not Path(SaveFolder).exists():
    #     print('doing')
    #     Path(SaveFolder).mkdir(parents=True, exist_ok=True)
    
    # plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)

In [ ]:
# THIS BLOCK RETRIEVES THE RELATIVE FREQUENCIES OF REFLECTIVITY VALUES TO PLOT IN THE CFAD

# what does the radar data use for the "fill value"?
FillValue = -32

MaxHeight = 10 # [km] maximum height you want to consider
ConsideredHeights = xgrid.z[np.where(xgrid.z<MaxHeight*1000)]

MinDBZ = -10 # [dBZ] minimum value of reflectivity you want to consider in the CFADs
MaxDBZ = 60 # [dBZ] minimum value of reflectivity you want to consider in the CFADs

NumHeights = np.size(ConsideredHeights) # the total number of altitudes in consideration

# find a floor and ceiling to the values of DBZ
LoEndDBZ = MinDBZ #int(np.floor(np.nanmin(flatZs)))
HiEndDBZ = MaxDBZ #int(np.ceil(np.nanmax(flatZs)))

NumDBZs = HiEndDBZ - LoEndDBZ # the total number of reflectivity [bins] in consideration

# create an empty array for storing frequencies and relative frequencies of reflectivity values
AbsCountsGrid = np.full([NumDBZs,NumHeights], np.nan)
NormCountsGrid = np.full([NumDBZs,NumHeights], np.nan)

# how many total grid cells in the radar data
TotalCells = np.size(xgrid.corrected_reflectivity)

# loop through each height and collect the relative frequencies
for zi in range(0, NumHeights):

    # store the flattened array of DBZ values
    flatZs = np.ndarray.flatten(np.array(xgrid.corrected_reflectivity[0,zi,:,:]))
    
    # replace the fill value with nans
    flatZs = np.where(flatZs == FillValue, np.nan, flatZs)
    
    # replace the low value with nans
    flatZs = np.where(flatZs < MinDBZ, np.nan, flatZs)
    
    # retrieve the data for a histogram
    
    counts, bins = np.histogram(flatZs[~np.isnan(flatZs)], bins=HiEndDBZ-LoEndDBZ, range=[LoEndDBZ, HiEndDBZ])

    # calculate the total cell frequencies and frequencies for that height
    AbsCounts  = counts * (1/TotalCells)
    NormCounts = counts * (1/np.sum(counts))
    
    # store away those counts and normalised counts in the grand CFAD data
    AbsCountsGrid[:,zi]  = AbsCounts
    NormCountsGrid[:,zi] = NormCounts

In [ ]:
# THIS BLOCK PLOTS A CFAD

Version = 'Absolute'

# based on the user's choice of version, pick the variable to plot, name the units its in, and set the colourbar limits for that variable
if (Version == 'Absolute'):
    PlotCounts = AbsCountsGrid * 100 # multiply by 100 to make it a percentage
    PlotUnit = '% of grid cells'
    MinVarVal = 0.00
    MaxVarVal = 0.10
elif (Version == 'Relative'):
    PlotCounts = NormCountsGrid  
    PlotUnit = 'per dBZ per km'
    MinVarVal = 0.00
    MaxVarVal = 0.25
else:
    sys.exit("please enter 'Absolute' or 'Relative' for Version")


# intervals of DBZ bins on the plot
DBZspacing = 1 # [dBZ]

DBZs = np.arange(LoEndDBZ,HiEndDBZ,DBZspacing) # (X COORDINATE ON THE PLOT) create a list of all possible DBZ values from -39 to 100 in steps of 1 
Heights = np.array(ConsideredHeights) * 0.001  # (Y COORDINATE ON THE PLOT) [converted to km] create a list of all of the altitudes where data are stored

# intervals of altitudes on the plot
HeightSpacing = 0.5 # [km] 

UprightFrequencies = np.transpose(PlotCounts) # (VALUES ON THE PLOT) transpose the stored relative frequencies of the reflectivities

# normalise frequencies to per unit DBZ per km of height
NormUprightFrequencies = UprightFrequencies * (1/DBZspacing) * (1/HeightSpacing)  

# PLOT A CFAD! (sort of)
fig, ax = plt.subplots(figsize=(8,6))

CFAD1 = pcolormeshC(DBZs , Heights, NormUprightFrequencies, ax=ax, cmap='nipy_spectral', vmin=MinVarVal, vmax=MaxVarVal)
plt.colorbar(CFAD1, ax=ax, label = Version + ' Frequency [' + PlotUnit + ']')

plt.xlim([MinDBZ, MaxDBZ])
plt.ylim([0,MaxHeight])

plt.title(Version + ' Frequencies of Corrected Reflectivity Values by Altitude\n for ' + RadarSiteName + ' Radar on ' + \
          RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
          str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC') 

ax.set_xlabel('Corrected Reflectivity [dBZ]')
ax.set_ylabel('Altitude [km]')

PlotVar  = 'corrected_reflectivity'

SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/CFAD/' + RadarIDno + '/' + RadarFileDate + '/'
SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + PlotVar + '_CFAD.png'

SavePath = SaveFolder + SaveFile

if not Path(SaveFolder).exists():
    print('Creating Folder: ' + SaveFolder)
    Path(SaveFolder).mkdir(parents=True, exist_ok=True)

plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
plt.close()

In [ ]:
# CFAD LOOP
# ADDED 2026-06-10T18:56UTC+10:00

# CHOOSE YOUR CFAD TYPE ('Absolute' or 'Relative')
Version = 'Absolute'

# what does the radar data use for the "fill value"?
FillValue = -32

# CHOOSE YOUR RADAR
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)

# name the radar site
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marabong'
else:
    RadarSiteName = 'Site ' + RadarIDno


# CHOOSE YOUR DAY
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 2
RadarDay   = 14

# CHOOSE YOUR HOUR and MINUTE
# houri = 00
# mini = 00

# LOOP OVER ALL SETS OF 5 MIN in the day
for houri in range(0,24):
    for mini in range(0,60,5):

        # add leading zeros for strings
        YYYY = str(RadarYear).zfill(4)
        MM = str(RadarMonth).zfill(2)
        DD = str(RadarDay).zfill(2)
        
        # write out the data in one string
        RadarFileDate  = YYYY + MM + DD 
        
        RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
        RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 
        
        
        print('working on reading the file for ' + RadarFileTimePrint)
        xgrid = xr.open_dataset('/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/' + \
                                RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')
        
        
        # THIS BLOCK RETRIEVES THE RELATIVE FREQUENCIES OF REFLECTIVITY VALUES TO PLOT IN THE CFAD
        
        MaxHeight = 10 # [km] maximum height you want to consider
        ConsideredHeights = xgrid.z[np.where(xgrid.z<MaxHeight*1000)]
        
        MinDBZ = -10 # [dBZ] minimum value of reflectivity you want to consider in the CFADs
        MaxDBZ = 60 # [dBZ] minimum value of reflectivity you want to consider in the CFADs
        
        NumHeights = np.size(ConsideredHeights) # the total number of altitudes in consideration
        
        # find a floor and ceiling to the values of DBZ
        LoEndDBZ = MinDBZ #int(np.floor(np.nanmin(flatZs)))
        HiEndDBZ = MaxDBZ #int(np.ceil(np.nanmax(flatZs)))
        
        NumDBZs = HiEndDBZ - LoEndDBZ # the total number of reflectivity [bins] in consideration
        
        # create an empty array for storing frequencies and relative frequencies of reflectivity values
        AbsCountsGrid = np.full([NumDBZs,NumHeights], np.nan)
        NormCountsGrid = np.full([NumDBZs,NumHeights], np.nan)
        
        # how many total grid cells in the radar data
        TotalCells = np.size(xgrid.corrected_reflectivity)
        
        # loop through each height and collect the relative frequencies
        for zi in range(0, NumHeights):
        
            # store the flattened array of DBZ values
            flatZs = np.ndarray.flatten(np.array(xgrid.corrected_reflectivity[0,zi,:,:]))
            
            # replace the fill value with nans
            flatZs = np.where(flatZs == FillValue, np.nan, flatZs)
            
            # replace the low value with nans
            flatZs = np.where(flatZs < MinDBZ, np.nan, flatZs)
            
            # retrieve the data for a histogram
            
            counts, bins = np.histogram(flatZs[~np.isnan(flatZs)], bins=HiEndDBZ-LoEndDBZ, range=[LoEndDBZ, HiEndDBZ])
        
            # calculate the total cell frequencies and frequencies for that height
            AbsCounts  = counts * (1/TotalCells)
            NormCounts = counts * (1/np.sum(counts))
            
            # store away those counts and normalised counts in the grand CFAD data
            AbsCountsGrid[:,zi]  = AbsCounts
            NormCountsGrid[:,zi] = NormCounts
        
        
        # start plotting!
        
        # based on the user's choice of version, pick the variable to plot, name the units its in, and set the colourbar limits for that variable
        if (Version == 'Absolute'):
            PlotCounts = AbsCountsGrid * 100 # multiply by 100 to make it a percentage
            PlotUnit = '% of grid cells'
            MinVarVal = 0.00
            MaxVarVal = 0.10
        elif (Version == 'Relative'):
            PlotCounts = NormCountsGrid  
            PlotUnit = 'per dBZ per km'
            MinVarVal = 0.00
            MaxVarVal = 0.25
        else:
            sys.exit("please enter 'Absolute' or 'Relative' for Version")
        
        
        # intervals of DBZ bins on the plot
        DBZspacing = 1 # [dBZ]
        
        DBZs = np.arange(LoEndDBZ,HiEndDBZ,DBZspacing) # (X COORDINATE ON THE PLOT) create a list of all possible DBZ values from -39 to 100 in steps of 1 
        Heights = np.array(ConsideredHeights) * 0.001  # (Y COORDINATE ON THE PLOT) [converted to km] create a list of all of the altitudes where data are stored
        
        # intervals of altitudes on the plot
        HeightSpacing = 0.5 # [km] 
        
        UprightFrequencies = np.transpose(PlotCounts) # (VALUES ON THE PLOT) transpose the stored relative frequencies of the reflectivities
        
        # normalise frequencies to per unit DBZ per km of height
        NormUprightFrequencies = UprightFrequencies * (1/DBZspacing) * (1/HeightSpacing)  
        
        # PLOT A CFAD! (sort of)
        fig, ax = plt.subplots(figsize=(8,6))
        
        CFAD1 = pcolormeshC(DBZs , Heights, NormUprightFrequencies, ax=ax, cmap='nipy_spectral', vmin=MinVarVal, vmax=MaxVarVal)
        plt.colorbar(CFAD1, ax=ax, label = Version + ' Frequency [' + PlotUnit + ']')
        
        plt.xlim([MinDBZ, MaxDBZ])
        plt.ylim([0,MaxHeight])
        
        plt.title(Version + ' Frequencies of Corrected Reflectivity Values by Altitude\n for ' + RadarSiteName + ' Radar on ' + \
                  RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
                  str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC') 
        
        ax.set_xlabel('Corrected Reflectivity [dBZ]')
        ax.set_ylabel('Altitude [km]')
        
        PlotVar  = 'corrected_reflectivity'
        
        SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/CFAD/' + RadarIDno + '/' + RadarFileDate + '/'
        SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + PlotVar + '_' + Version + 'CFAD.png'
        
        SavePath = SaveFolder + SaveFile
        
        if not Path(SaveFolder).exists():
            print('Creating Folder: ' + SaveFolder)
            Path(SaveFolder).mkdir(parents=True, exist_ok=True)
        
        plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
        plt.close()

In [ ]:
# GIF MAKER
# FOR CFADS

SavedFolder = '/scratch/v46/sg3241/tmp/pngImages/CFAD/' + RadarIDno + '/' + RadarFileDate + '/'
                
# LOADING IMAGES
files = sorted(os.listdir(SavedFolder)) # takes all of the files in the folder in the order they are named
images = [
    Image.open(os.path.join(SavedFolder, f))
    for f in files
    if f.endswith(Version + 'CFAD.png')
]

# SaveFolder = '/cratch/v46/sg3241/tmp/pngImages/Vert/22/20240214/'

#         SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/CFAD/' + RadarIDno + '/' + RadarFileDate + '/'
#         SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + PlotVar + '_' + Version + 'CFAD.png'

GIFsaveFolder = '/scratch/v46/sg3241/tmp/gifImages/CFAD/' + RadarIDno + '/' + RadarFileDate + '/'
GIFsaveFile   = RadarIDno + '_' + RadarFileDate + '_' + PlotVar + '_' + Version + 'CFAD.gif'

GIFsavePath = GIFsaveFolder + GIFsaveFile

if not Path(GIFsaveFolder).exists():
    Path(GIFsaveFolder).mkdir(parents=True, exist_ok=True)

# Save as looping GIF
images[0].save(
    GIFsavePath,
    save_all=True,
    append_images=images[1:],
    duration=200,    # ms per frame
    loop=0,          # 0 = loop forever
)
print('Saved GIF for ' + RadarFileDatePrint)

In [ ]:
Version + '.png'